In [0]:
-- Gold Layer containing megerd data from silver layer for ML Model based ready data

-- This is the primary ML dataset (best for demand, late delivery, profit, stockout, etc.)

-- main.default.gold_sales_ml_clean


CREATE OR REPLACE TABLE main.default.gold_sales_ml_clean AS
WITH supply_chain_agg AS (
    SELECT
        product_type,
        AVG(defect_rates) AS avg_defect_rate,
        MAX(defect_rates) AS max_defect_rate,
        AVG(shipping_costs) AS avg_shipping_cost,
        AVG(shipping_times) AS avg_shipping_time,
        AVG(lead_times) AS avg_lead_time,
        AVG(stock_levels) AS avg_stock_level,
        COUNT(DISTINCT sku) AS num_skus,
        COUNT(DISTINCT supplier_name) AS num_suppliers
    FROM main.default.silver_supply_chain_data
    GROUP BY product_type
),
gold_raw AS (
    SELECT
        s.order_id,
        s.order_date,
        s.shipment_date,
        s.product_id,
        s.customer_id,
        s.customer_segment,
        s.sales,
        s.quantity,
        s.profit,
        s.is_late_delivery,
        s.shipping_mode,
        s.market,
        s.department,
        s.class,

        i.lead_time,
        i.current_stock,
        i.reorder_point,
        i.avg_order,
        i.safety_stock,

        sc.avg_defect_rate,
        sc.max_defect_rate,
        sc.avg_shipping_cost,
        sc.avg_shipping_time,
        sc.avg_lead_time AS sc_avg_lead_time,
        sc.avg_stock_level,
        sc.num_skus,
        sc.num_suppliers

    FROM main.default.silver_sales_shipment_data s
    LEFT JOIN main.default.silver_inventory_stock_data i
        ON s.product_id = i.product_id
    LEFT JOIN supply_chain_agg sc
        ON (s.department = 'Health and Beauty' AND sc.product_type IN ('haircare', 'skincare', 'cosmetics'))
)

SELECT
    order_id,
    order_date,
    shipment_date,
    product_id,
    customer_id,
    customer_segment,
    sales,
    quantity,
    profit,
    is_late_delivery,
    shipping_mode,
    market,
    department,
    class,

    COALESCE(lead_time, 0)        AS lead_time,
    COALESCE(current_stock, 0)    AS current_stock,
    COALESCE(gr.reorder_point, 0)    AS reorder_point,
    COALESCE(gr.avg_order, 0)        AS avg_order,
    COALESCE(gr.safety_stock, 0)     AS safety_stock,

    COALESCE(gr.avg_defect_rate, 0)  AS avg_defect_rate,
    COALESCE(gr.max_defect_rate, 0)  AS max_defect_rate,
    COALESCE(gr.avg_shipping_cost, 0) AS avg_shipping_cost,
    COALESCE(gr.avg_shipping_time, 0) AS avg_shipping_time,
    COALESCE(gr.sc_avg_lead_time, 0) AS sc_avg_lead_time,
    COALESCE(gr.avg_stock_level, 0)  AS avg_stock_level,
    COALESCE(gr.num_skus, 0)         AS num_skus,
    COALESCE(gr.num_suppliers, 0)    AS num_suppliers,

    CASE WHEN gr.lead_time IS NULL THEN 1 ELSE 0 END AS lead_time_missing,
    CASE WHEN gr.current_stock IS NULL THEN 1 ELSE 0 END AS stock_missing,
    CASE WHEN gr.avg_defect_rate IS NULL THEN 1 ELSE 0 END AS defect_missing,
    CASE WHEN gr.avg_shipping_cost IS NULL THEN 1 ELSE 0 END AS shipping_cost_missing

FROM gold_raw gr;

SELECT * FROM main.default.gold_sales_ml_clean;

----------------------------------------------------

-- main.default.gold_inventory_ml_clean

CREATE OR REPLACE TABLE main.default.gold_inventory_ml_clean AS
WITH gold_raw AS (
    SELECT
        inv.date,
        inv.store_id,
        CAST(inv.product_id AS STRING) AS product_id,
        inv.sales AS store_sales,
        inv.stock AS store_stock,

        i.lead_time,
        i.current_stock,
        i.reorder_point,
        i.avg_order,
        i.safety_stock

    FROM main.default.silver_retail_store_inventory inv
    LEFT JOIN main.default.silver_inventory_stock_data i
        ON CAST(inv.product_id AS STRING) = CAST(i.product_id AS STRING)
)

SELECT
    date,
    store_id,
    product_id,
    store_sales,
    store_stock,

    COALESCE(lead_time, 0)        AS lead_time,
    COALESCE(current_stock, 0)    AS current_stock,
    COALESCE(reorder_point, 0)    AS reorder_point,
    COALESCE(avg_order, 0)        AS avg_order,
    COALESCE(safety_stock, 0)     AS safety_stock,

    CASE WHEN lead_time IS NULL THEN 1 ELSE 0 END AS lead_time_missing,
    CASE WHEN current_stock IS NULL THEN 1 ELSE 0 END AS stock_missing

FROM gold_raw;


SELECT * FROM main.default.gold_inventory_ml_clean;

-------------------------------------------------------------

-- main.default.gold_inventory_features

CREATE OR REPLACE TABLE main.default.gold_inventory_features AS
SELECT 
    date,
    store_id,
    product_id,
    COALESCE(store_sales, 0) AS store_sales,
    COALESCE(current_stock, 0) AS current_stock,
    COALESCE(reorder_point, 0) AS reorder_point,
    
    -- 30-day rolling features (NULL -> 0)
    COALESCE(SUM(store_sales) OVER (PARTITION BY store_id, product_id ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW), 0) AS sales_30d,
    COALESCE(AVG(store_sales) OVER (PARTITION BY store_id, product_id ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW), 0) AS avg_sales_30d,
    COALESCE(STDDEV(store_sales) OVER (PARTITION BY store_id, product_id ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW), 0) AS stddev_sales_30d,
    
    -- Inventory ratios (NULL -> 0)
    COALESCE(ROUND(current_stock / NULLIF(avg_order, 0), 2), 0) AS days_inventory_outstanding,
    CASE 
        WHEN current_stock IS NOT NULL AND reorder_point IS NOT NULL AND current_stock < reorder_point THEN 1 
        ELSE 0 
    END AS stock_below_reorder,
    
    -- Time features
    DAYOFWEEK(date) AS day_of_week,
    MONTH(date) AS month,
    QUARTER(date) AS quarter,
    YEAR(date) AS year,
    
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_DATE() AS feature_date
FROM main.default.gold_inventory_ml_clean
WHERE date IS NOT NULL AND store_sales IS NOT NULL;


SELECT * FROM main.default.gold_inventory_features;


----------------------------------------------------------------

-- main.default.gold_delivery_features

CREATE OR REPLACE TABLE main.default.gold_delivery_features AS
SELECT 
    order_id,
    order_date,
    shipment_date,
    product_id,
    customer_id,
    customer_segment,
    sales,
    quantity,
    profit,
    is_late_delivery,
    shipping_mode,
    market,
    lead_time,
    
    -- Time-based features
    DATEDIFF(DAY, order_date, shipment_date) AS order_processing_days,
    DATEDIFF(DAY, shipment_date, CURRENT_DATE()) AS days_since_shipment,
    
    -- Customer 30-day features
    AVG(sales) OVER (PARTITION BY customer_id ORDER BY order_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS avg_order_value_30d,
    COUNT(*) OVER (PARTITION BY customer_id ORDER BY order_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS num_orders_30d,
    
    -- Shipping mode 30-day stats
    AVG(lead_time) OVER (PARTITION BY shipping_mode ORDER BY order_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS avg_lead_time_by_mode,
    AVG(is_late_delivery) OVER (PARTITION BY shipping_mode ORDER BY order_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS late_delivery_rate_by_mode,
    
    -- Categories
    CASE WHEN sales > 300 THEN 1 ELSE 0 END AS is_high_value,
    CASE WHEN quantity > 5 THEN 1 ELSE 0 END AS is_bulk_order,
    CASE WHEN profit > 0 THEN 1 ELSE 0 END AS is_profitable,
    ROUND(profit / NULLIF(sales, 0), 3) AS profit_margin,
    
    -- Time features (from order_date)
    DAYOFWEEK(order_date) AS day_of_week,
    MONTH(order_date) AS month,
    QUARTER(order_date) AS quarter,
    YEAR(order_date) AS year,
    
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_DATE() AS feature_date
FROM main.default.gold_sales_ml_clean
WHERE order_id IS NOT NULL;


SELECT * FROM main.default.gold_delivery_features;


--------------------------------------------------------

-- main.default.gold_combined_features_ml

CREATE OR REPLACE TABLE main.default.gold_combined_features_ml AS
SELECT 
    is_late_delivery, 
    order_id,
    product_id,
    customer_id,
    customer_segment,
    sales,
    quantity,
    profit,
    shipping_mode,
    market,
    lead_time,
    order_processing_days,
    avg_order_value_30d,
    num_orders_30d,
    avg_lead_time_by_mode,
    late_delivery_rate_by_mode,
    is_high_value,
    is_bulk_order,
    is_profitable,
    profit_margin,
    day_of_week,
    month,
    quarter,
    year,
    CURRENT_DATE() AS feature_extraction_date
FROM main.default.gold_delivery_features
WHERE feature_date = CURRENT_DATE();

SELECT * FROM main.default.gold_combined_features_ml;